# 📖 Notebook 3: Route Matching & Segment Leaderboards

Strava's killer feature: **segments**. A segment is a specific stretch of road
that athletes race on. When your GPS trace passes through a segment, Strava
automatically records your time and ranks you on a leaderboard.

In this notebook we build the segment system end-to-end.

## Learning Objectives

By the end of this notebook, you'll understand:
- What segments are and how they're defined
- How to detect when a GPS route crosses a segment (**route matching**)
- How **Redis Sorted Sets** power O(log N) leaderboards
- How to filter leaderboards by city, country, and time range
- The scaling trade-offs of different leaderboard approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/strava
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `strava_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import math
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "strava_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def haversine(lat1, lon1, lat2, lon2):
    """Distance in metres between two GPS points."""
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🏁 What Is a Segment?

A **segment** is a stretch of road defined by a start point and an end point.
Think of it as a race course — everyone who passes through it gets timed.

Examples:
- *Embarcadero Sprint* — 850 m along the SF waterfront
- *Golden Gate Park Climb* — 2.8 km through the park
- *Central Park North Loop* — 1.2 km in NYC

A **segment effort** is one athlete's attempt at a segment.
The fastest effort wins the **leaderboard** (King/Queen of the Mountain).

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# List all segments
cur.execute("SELECT id, name, type, distance_m, city FROM segments ORDER BY id")
segments = cur.fetchall()

print("Available Segments:")
print(f"{'ID':>4} {'Name':<30} {'Type':<6} {'Distance':>10} {'City':<15}")
print("-" * 70)
for s in segments:
    print(f"{s['id']:>4} {s['name']:<30} {s['type']:<6} {s['distance_m']:>8.0f} m  {s['city']:<15}")

# Show existing efforts for the Embarcadero Sprint
print()
cur.execute("""
    SELECT se.id, u.username, se.elapsed_s, s.name
    FROM segment_efforts se
    JOIN users u ON se.user_id = u.id
    JOIN segments s ON se.segment_id = s.id
    WHERE se.segment_id = 1
    ORDER BY se.elapsed_s
    LIMIT 5;
""")
efforts = cur.fetchall()

print(f"Embarcadero Sprint — Top efforts:")
for i, e in enumerate(efforts, 1):
    mins = e['elapsed_s'] // 60
    secs = e['elapsed_s'] % 60
    print(f"  #{i} {e['username']:<10} {mins}:{secs:02d}")

conn.close()

## 🔍 Route Matching: Did You Cross a Segment?

When an activity is uploaded, we need to check if the GPS trace passes through
any known segments. This is **route matching**.

### Simple approach (good enough for interviews):

1. For each segment, find the route point **closest to the segment start**, then the
   route point **closest to the segment end** among the points that came after it
2. Both have to be within a threshold distance (e.g., 100 metres) to count as a match
3. If both match, the effort time is the gap between those two points' timestamps

Note the "closest", not "first within the threshold". A GPS trace usually has several
points inside a 100 m ball around an endpoint; taking the first one you stumble across
starts the clock early and stops it late, and — worse — gives a different answer than
the SQL version of the same query. Ranking by distance makes the choice deterministic.

```
Segment: Embarcadero Sprint
  Start: (37.7955, -122.3935)
  End:   (37.7899, -122.3860)

Your GPS trace:
  Point 1: (37.7954, -122.3936) ← within 50 m of segment start ✅
  Point 2: (37.7948, -122.3925)
  ...
  Point 9: (37.7900, -122.3861) ← within 50 m of segment end ✅

→ Matched! Effort time = timestamp(point 9) - timestamp(point 1)
```

In [ ]:
MATCH_THRESHOLD_M = 100   # how close a GPS point must be to a segment endpoint
PATH_TOLERANCE = 0.25     # how far the distance travelled may stray from the segment's


def track_distance(points, i, j):
    """Distance actually covered along the trace, from point i to point j."""
    return sum(haversine(points[k]['latitude'], points[k]['longitude'],
                         points[k + 1]['latitude'], points[k + 1]['longitude'])
               for k in range(i, j))


def nearest_point(points, lat, lon, start_index=0):
    """Index of the trace point closest to (lat, lon), searching from start_index."""
    best_i, best_d = None, None
    for i in range(start_index, len(points)):
        d = haversine(points[i]['latitude'], points[i]['longitude'], lat, lon)
        if best_d is None or d < best_d:
            best_i, best_d = i, d
    return best_i, best_d


def match_trace_to_segment(points, seg, require_path_match=False):
    """
    Does this ordered GPS trace contain an effort on `seg`?
    Returns a match dict or None.

    require_path_match=False is the endpoint-only rule described above. Turning it on
    also requires the distance *travelled* between the two matched points to look like
    the segment's own length -- see the next section for why you want that.
    """
    if len(points) < 2:
        return None

    start_i, start_d = nearest_point(points, seg['start_lat'], seg['start_lon'])
    if start_d > MATCH_THRESHOLD_M:
        return None
    # The end has to come *after* the start -- otherwise we'd happily match a trace
    # run in the wrong direction.
    end_i, end_d = nearest_point(points, seg['end_lat'], seg['end_lon'], start_i + 1)
    if end_i is None or end_d > MATCH_THRESHOLD_M:
        return None

    travelled = track_distance(points, start_i, end_i)
    ratio = travelled / seg['distance_m']
    if require_path_match and abs(ratio - 1.0) > PATH_TOLERANCE:
        return None

    elapsed = (points[end_i]['recorded_at']
               - points[start_i]['recorded_at']).total_seconds()
    return {
        'segment_id': seg['id'],
        'segment_name': seg['name'],
        'elapsed_s': int(elapsed),
        'start_seq': points[start_i]['seq'],
        'end_seq': points[end_i]['seq'],
        'start_dist_m': start_d,
        'end_dist_m': end_d,
        'travelled_m': travelled,
        'path_ratio': ratio,
    }


def match_route_to_segments(activity_id, require_path_match=False):
    """Check a stored activity's GPS trace against every segment of the same type."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT seq, latitude, longitude, recorded_at
        FROM route_points
        WHERE activity_id = %s
        ORDER BY seq;
    """, (activity_id,))
    points = cur.fetchall()

    cur.execute("SELECT type, user_id FROM activities WHERE id = %s", (activity_id,))
    activity = cur.fetchone()

    # Only same-type segments: you don't get a running PB for cycling past it.
    cur.execute("SELECT * FROM segments WHERE type = %s", (activity['type'],))
    segments = cur.fetchall()
    conn.close()

    matches = []
    for seg in segments:
        m = match_trace_to_segment(points, seg, require_path_match)
        if m:
            m['user_id'] = activity['user_id']
            matches.append(m)
    return matches


# Match Alice's Embarcadero run (activity 1)
matches = match_route_to_segments(activity_id=1)

print("Route matching for Activity #1 (Alice's Embarcadero Run):")
print()
for m in matches:
    print(f"  ✅ Matched: {m['segment_name']}")
    print(f"     Points {m['start_seq']} → {m['end_seq']} "
          f"({m['start_dist_m']:.0f} m / {m['end_dist_m']:.0f} m from the endpoints)")
    print(f"     Travelled {m['travelled_m']:.0f} m against a "
          f"{m['path_ratio']:.2f}× segment length")
    print(f"     Effort time: {m['elapsed_s'] // 60}:{m['elapsed_s'] % 60:02d}")
if not matches:
    print("  No segments matched.")

# The stored effort was computed from this same trace, so recomputing it has to
# reproduce it. If this drifts, either the matcher or the seed data is wrong.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT elapsed_s FROM segment_efforts WHERE segment_id = 1 AND activity_id = 1
""")
stored_effort = cur.fetchone()[0]
conn.close()

assert len(matches) == 1, f"expected exactly 1 segment match for activity 1, got {len(matches)}"
assert matches[0]['segment_id'] == 1, "activity 1 should match the Embarcadero Sprint"
assert matches[0]['elapsed_s'] == stored_effort, (
    f"recomputed effort {matches[0]['elapsed_s']} s != stored effort {stored_effort} s. "
    f"If you are on an old Postgres volume, re-seed: docker compose down -v && up -d"
)

print()
print(f"💡 Recomputed effort matches segment_efforts.elapsed_s ({stored_effort} s).")
print("   In production this matching runs automatically when activities are uploaded;")
print("   PostGIS ST_DWithin keeps the proximity check cheap over millions of points.")

In [ ]:
# PostGIS version: the same matching rule, expressed as spatial SQL.
#
# The naive way to write this is `start_matches JOIN end_matches ON segment_id`, which
# is a CROSS JOIN in disguise: every point near the start pairs with every point near
# the end. Alice has two points inside the 100 m start ball (seq 1 and 2) and two inside
# the end ball (seq 9 and 10), so that version reports FOUR efforts -- 3:30, 4:00, 4:00
# and 4:30 -- for one run past one segment, and none of them need agree with the Python
# matcher above.
#
# DISTINCT ON ... ORDER BY dist picks the single nearest candidate per segment, which
# is the rule we implemented in Python.
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    WITH pts AS (
        SELECT seq, geom, recorded_at
        FROM route_points
        WHERE activity_id = 1
    ),
    segs AS (
        SELECT id, name, start_lat, start_lon, end_lat, end_lon
        FROM segments
        WHERE type = 'RUN'
    ),
    -- nearest route point to each segment's start (ST_DWithin uses the GIST index)
    starts AS (
        SELECT DISTINCT ON (s.id)
               s.id AS segment_id, s.name, p.seq, p.recorded_at,
               ST_Distance(p.geom, ST_Point(s.start_lon, s.start_lat)::geography) AS dist
        FROM segs s
        JOIN pts p ON ST_DWithin(p.geom,
                                 ST_Point(s.start_lon, s.start_lat)::geography, 100)
        ORDER BY s.id, dist
    ),
    -- nearest route point to the end, restricted to points AFTER the matched start
    ends AS (
        SELECT DISTINCT ON (st.segment_id)
               st.segment_id, p.seq, p.recorded_at,
               ST_Distance(p.geom, ST_Point(s.end_lon, s.end_lat)::geography) AS dist
        FROM starts st
        JOIN segs s ON s.id = st.segment_id
        JOIN pts p ON p.seq > st.seq
                  AND ST_DWithin(p.geom,
                                 ST_Point(s.end_lon, s.end_lat)::geography, 100)
        ORDER BY st.segment_id, dist
    )
    SELECT st.segment_id, st.name,
           st.seq AS start_seq, e.seq AS end_seq,
           EXTRACT(EPOCH FROM (e.recorded_at - st.recorded_at))::int AS elapsed_s
    FROM starts st
    JOIN ends e ON e.segment_id = st.segment_id
    ORDER BY st.segment_id;
""")
pg_matches = cur.fetchall()
conn.close()

print("PostGIS route matching (pure SQL):")
for m in pg_matches:
    print(f"  ✅ {m['name']}: points {m['start_seq']}→{m['end_seq']}, "
          f"time {m['elapsed_s'] // 60}:{m['elapsed_s'] % 60:02d}")

# Two implementations of one rule must agree, or one of them is the bug.
py_by_id = {m['segment_id']: m for m in matches}
pg_by_id = {m['segment_id']: m for m in pg_matches}
assert set(py_by_id) == set(pg_by_id), (
    f"Python matched segments {sorted(py_by_id)} but SQL matched {sorted(pg_by_id)}"
)
for sid, pg in pg_by_id.items():
    py = py_by_id[sid]
    assert (pg['start_seq'], pg['end_seq'], pg['elapsed_s']) == \
           (py['start_seq'], py['end_seq'], py['elapsed_s']), (
        f"segment {sid}: SQL says {pg['start_seq']}→{pg['end_seq']} "
        f"({pg['elapsed_s']} s), Python says {py['start_seq']}→{py['end_seq']} "
        f"({py['elapsed_s']} s)"
    )

print()
print("💡 One row per segment, identical to the Python matcher — and PostGIS uses the")
print("   GIST spatial index for the proximity test, so it scales to millions of points.")

## 🎯 Bad → Best: Endpoints Are Not a Route

The matcher above asks one question: *did you pass near the start, and later near the
end?* That is a surprisingly weak question. It says nothing about **what you did in
between**.

Two athletes both trigger a match on the Embarcadero Sprint:

- one runs the waterfront, hugging the segment the whole way — GPS wobble and all;
- one starts at the same spot, runs three blocks inland, loops around, comes back and
  finishes at the same spot.

The second athlete gets a leaderboard time for a segment they never ran. In real life
this is not hypothetical — it is how people accidentally (and deliberately) end up on
segment leaderboards for roads they drove, and it is why "did the endpoints match" is
not a shippable rule.

**Bad**: match on the two endpoints alone.
**Best**: also check that the distance *actually travelled* between the matched points
resembles the segment's own length. A wobbly GPS trace measures a few percent long; a
detour measures multiples. One cheap ratio separates them, and it needs nothing beyond
the `distance_m` already on the segment row.

The tolerance matters and is a real trade-off: too tight and honest efforts on noisy
urban GPS get rejected, too loose and the detour slips through. A production system
goes further and stores the segment's full polyline, requiring the trace to stay near
*every* vertex (a discrete Fréchet-style check) — strictly better, and strictly more
data to carry. We show the cheap version because it catches the common case.

In [ ]:
import random
from datetime import datetime, timedelta

random.seed(2718)   # seeded: the two synthetic traces below are reproducible

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT * FROM segments WHERE id = 1")
sprint = cur.fetchone()
conn.close()

M_PER_DEG_LAT = 111_320.0
GPS_SIGMA_M = 8.0          # urban GPS, between tall buildings


def as_trace(coords, step_s=15):
    """Turn [(lat, lon), ...] into the point dicts the matcher expects."""
    t0 = datetime(2026, 4, 20, 8, 0, 0)
    return [{'seq': i + 1, 'latitude': lat, 'longitude': lon,
             'recorded_at': t0 + timedelta(seconds=i * step_s)}
            for i, (lat, lon) in enumerate(coords)]


def jitter(lat, lon):
    """Add realistic GPS noise to one point."""
    lon_scale = M_PER_DEG_LAT * math.cos(math.radians(lat))
    return (lat + random.gauss(0, GPS_SIGMA_M) / M_PER_DEG_LAT,
            lon + random.gauss(0, GPS_SIGMA_M) / lon_scale)


def interpolate(a, b, n):
    return [(a[0] + (b[0] - a[0]) * i / n, a[1] + (b[1] - a[1]) * i / n)
            for i in range(n + 1)]

start = (sprint['start_lat'], sprint['start_lon'])
end = (sprint['end_lat'], sprint['end_lon'])

# (a) An honest effort: down the waterfront, with 8 m of GPS wobble on every point.
honest = as_trace([jitter(lat, lon) for lat, lon in interpolate(start, end, 30)])

# (b) A detour: same two endpoints, ~600 m inland and back in between.
inland = (start[0] + 0.006, start[1] - 0.004)
detour = as_trace(interpolate(start, inland, 10) + interpolate(inland, end, 20)[1:])

print(f"Segment: {sprint['name']} — official length {sprint['distance_m']:.0f} m")
print()
print(f"{'trace':<10} {'endpoint-only':<16} {'travelled':>11} {'ratio':>7}  {'with path check':<16}")
print("-" * 68)

results = {}
for name, trace in (('honest', honest), ('detour', detour)):
    loose = match_trace_to_segment(trace, sprint, require_path_match=False)
    strict = match_trace_to_segment(trace, sprint, require_path_match=True)
    results[name] = (loose, strict)
    print(f"{name:<10} {'MATCH ✅' if loose else 'no match':<16} "
          f"{loose['travelled_m']:>9.0f} m {loose['path_ratio']:>7.2f}  "
          f"{'MATCH ✅' if strict else 'REJECTED ❌':<16}")

(honest_loose, honest_strict) = results['honest']
(detour_loose, detour_strict) = results['detour']

# 1. The failure must actually reproduce: endpoints alone accept the detour.
assert detour_loose is not None, (
    "the endpoint-only matcher rejected the detour, so this cell is not "
    "demonstrating the bug it claims to"
)
assert detour_loose['path_ratio'] > 1 + PATH_TOLERANCE, (
    f"detour only measured {detour_loose['path_ratio']:.2f}× the segment length -- "
    f"make the detour longer or this is not a meaningful test"
)
# 2. The fix must reject it...
assert detour_strict is None, "the path check let the detour through"
# 3. ...without rejecting an honest, noisy effort.
assert honest_strict is not None, (
    f"the path check rejected an honest effort measuring "
    f"{honest_loose['path_ratio']:.2f}× the segment -- PATH_TOLERANCE is too tight "
    f"for {GPS_SIGMA_M:.0f} m of GPS noise"
)
assert honest_strict['elapsed_s'] == honest_loose['elapsed_s'], \
    "the path check should not change the time of an accepted effort"

print()
print(f"💡 GPS noise made the honest trace measure "
      f"{(honest_loose['path_ratio'] - 1) * 100:+.0f}% — well inside the "
      f"±{PATH_TOLERANCE * 100:.0f}% tolerance.")
print(f"   The detour measured {detour_loose['path_ratio']:.1f}× the segment and is out.")
print("   Note what this still does NOT catch: a detour that happens to be the right")
print("   length, or a trace that runs the segment backwards on a there-and-back road.")
print("   That is what the full-polyline check buys you.")

## 🏆 Leaderboards with Redis Sorted Sets

A **Sorted Set** in Redis is a collection where each member has a **score**.
Members are automatically sorted by score, and you can query ranges efficiently.

Perfect for leaderboards!

```
Key:    leaderboard:segment:1         (Embarcadero Sprint)
Member: user_id                       (who)
Score:  best elapsed time in seconds  (lower is better)
```

### Key Operations (all O(log N)):

| Operation | Redis Command | What It Does |
|-----------|--------------|---------------|
| Add/update | `ZADD` | Add a score or update if lower |
| Top N | `ZRANGE` | Get the N best scores |
| My rank | `ZRANK` | Find where I am in the ranking |
| My score | `ZSCORE` | Get my best time |

In [ ]:
r = get_redis()

def build_leaderboard_from_db(segment_id):
    """
    Build a Redis leaderboard from existing segment efforts in Postgres.
    For each user, only their best (lowest) time counts.
    """
    conn = get_db()
    cur = conn.cursor()

    # Get best effort per user for this segment
    cur.execute("""
        SELECT user_id, MIN(elapsed_s) AS best_time
        FROM segment_efforts
        WHERE segment_id = %s
        GROUP BY user_id;
    """, (segment_id,))

    key = f"leaderboard:segment:{segment_id}"
    r.delete(key)

    count = 0
    for user_id, best_time in cur.fetchall():
        # Score = elapsed_s (lower is better, so lower score = higher rank)
        r.zadd(key, {str(user_id): best_time})
        count += 1

    conn.close()
    return count

# Build leaderboards for all segments
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, name FROM segments")
all_segments = cur.fetchall()
conn.close()

for seg_id, seg_name in all_segments:
    n = build_leaderboard_from_db(seg_id)
    print(f"  Built leaderboard for '{seg_name}': {n} athletes")


# Redis is a cache of what Postgres already knows. Prove it starts out agreeing.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT segment_id, user_id, MIN(elapsed_s)
    FROM segment_efforts GROUP BY segment_id, user_id
""")
expected = {(sid, str(uid)): float(t) for sid, uid, t in cur.fetchall()}
conn.close()
in_redis = {(sid, uid): score
            for sid, _ in all_segments
            for uid, score in r.zrange(f"leaderboard:segment:{sid}", 0, -1, withscores=True)}
assert in_redis == expected, (
    f"Redis leaderboards disagree with Postgres right after building them from it: "
    f"{len(set(expected) ^ set(in_redis))} member(s) differ"
)
print()
print("💡 Redis and Postgres agree. Open RedisInsight to see the sorted sets!")

In [ ]:
def get_leaderboard(segment_id, top_n=10):
    """Get the top N athletes for a segment from Redis."""
    key = f"leaderboard:segment:{segment_id}"
    # ZRANGE with WITHSCORES returns [(member, score), ...]
    # Lower score = faster time = better rank
    results = r.zrange(key, 0, top_n - 1, withscores=True)
    
    # Look up usernames
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    leaderboard = []
    for user_id_str, score in results:
        cur.execute("SELECT username, display_name, city FROM users WHERE id = %s",
                    (int(user_id_str),))
        user = cur.fetchone()
        elapsed = int(score)
        leaderboard.append({
            'user_id': int(user_id_str),
            'username': user['username'],
            'display_name': user['display_name'],
            'city': user['city'],
            'elapsed_s': elapsed,
            'time_str': f"{elapsed // 60}:{elapsed % 60:02d}",
        })
    conn.close()
    return leaderboard

# Show leaderboard for each segment
for seg_id, seg_name in all_segments:
    lb = get_leaderboard(seg_id, top_n=5)
    print(f"🏆 {seg_name} — Top 5:")
    for i, entry in enumerate(lb, 1):
        crown = '👑' if i == 1 else '  '
        print(f"  {crown} #{i} {entry['display_name']:<20} {entry['time_str']:>6} "
              f"  ({entry['city']})")
    print()

In [ ]:
# Recording an effort: write-through, and let Redis do the comparison.
def record_segment_effort(segment_id, activity_id, user_id, elapsed_s):
    """
    Persist an effort and update the leaderboard.

    Two things worth copying from this function:

    1. **Postgres first.** Redis is a derived view -- anything that rebuilds it from
       Postgres (like build_filtered_leaderboards below) will erase an effort that
       only ever existed in Redis. A leaderboard entry with no row behind it is the
       same class of bug as marking a payment captured without writing the ledger.
    2. **ZADD ... lt=True instead of read-then-write.** `ZSCORE`, compare in Python,
       then `ZADD` is a lost-update race: two devices uploading at the same moment
       both read the old best and the slower one can land last. `lt=True` lowers the
       score or does nothing, atomically, in a single round trip.
    """
    conn = get_db()
    cur = conn.cursor()
    # One effort per (segment, activity) keeps re-running this cell idempotent.
    cur.execute("""
        DELETE FROM segment_efforts WHERE segment_id = %s AND activity_id = %s
    """, (segment_id, activity_id))
    cur.execute("""
        INSERT INTO segment_efforts (segment_id, activity_id, user_id, elapsed_s, started_at)
        VALUES (%s, %s, %s, %s, NOW())
    """, (segment_id, activity_id, user_id, elapsed_s))
    conn.close()

    key = f"leaderboard:segment:{segment_id}"
    previous_best = r.zscore(key, str(user_id))   # for display only -- advisory
    r.zadd(key, {str(user_id): elapsed_s}, lt=True)
    new_best = r.zscore(key, str(user_id))
    rank = r.zrank(key, str(user_id))

    return {
        'personal_best': previous_best is None or elapsed_s < previous_best,
        'rank': rank + 1 if rank is not None else None,
        'time': f"{elapsed_s // 60}:{elapsed_s % 60:02d}",
        'best_now_s': int(new_best),
        'previous_best': ('N/A' if previous_best is None
                          else f"{int(previous_best) // 60}:{int(previous_best) % 60:02d}"),
    }


# Reset the demo so the cell tells the same story on every run. (Real systems never
# delete efforts; this is only here to keep the notebook re-runnable.)
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM segment_efforts WHERE segment_id = 1 AND activity_id = 3")
conn.close()
r.zrem("leaderboard:segment:1", "3")

# Carol (user 3) posts a 3:15 on the Embarcadero Sprint, from her activity #3.
CAROL_TIME_S = 195
result = record_segment_effort(segment_id=1, activity_id=3, user_id=3,
                               elapsed_s=CAROL_TIME_S)
print(f"Carol just finished the Embarcadero Sprint in {result['time']}!")
print(f"  Personal best: {'YES 🎉' if result['personal_best'] else 'No'}")
print(f"  Previous best: {result['previous_best']}")
print(f"  Current rank:  #{result['rank']}")
print()

# Show updated leaderboard
lb = get_leaderboard(segment_id=1, top_n=5)
print("Updated Embarcadero Sprint leaderboard:")
for i, entry in enumerate(lb, 1):
    crown = '👑' if i == 1 else '  '
    new = ' ← NEW!' if entry['username'] == 'carol' else ''
    print(f"  {crown} #{i} {entry['display_name']:<20} {entry['time_str']:>6}{new}")

# The leaderboard must reflect the write, and the rank must be derivable from the
# data rather than from what we hope Redis did.
assert r.zscore("leaderboard:segment:1", "3") == CAROL_TIME_S, \
    "Carol's effort did not make it into the sorted set"
conn = get_db()
cur = conn.cursor()
# Count athletes strictly faster, and those tied on the same second. Seeded times
# are whole seconds, so a tie is not exotic -- Redis breaks it lexicographically by
# member, which is arbitrary but stable. Assert the band, not one exact rank.
cur.execute("""
    SELECT COUNT(*) FILTER (WHERE b.best < %s),
           COUNT(*) FILTER (WHERE b.best = %s)
    FROM (
        SELECT user_id, MIN(elapsed_s) AS best FROM segment_efforts
        WHERE segment_id = 1 AND user_id <> 3 GROUP BY user_id
    ) b
""", (CAROL_TIME_S, CAROL_TIME_S))
faster, tied = cur.fetchone()
cur.execute("""
    SELECT elapsed_s FROM segment_efforts WHERE segment_id = 1 AND activity_id = 3
""")
persisted = cur.fetchone()
conn.close()
assert persisted is not None and persisted[0] == CAROL_TIME_S, (
    "the effort was written to Redis but not to Postgres -- the next rebuild from "
    "Postgres would silently erase it"
)
assert faster + 1 <= result['rank'] <= faster + tied + 1, (
    f"Redis ranks Carol #{result['rank']}, but Postgres has {faster} athletes strictly "
    f"faster and {tied} tied -- her rank must be between #{faster + 1} and "
    f"#{faster + tied + 1}"
)

print()
print(f"💡 Carol sits behind {faster} faster athlete(s), {tied} tied on the second.")
print("   The row is in Postgres, so rebuilding the leaderboard from the database")
print("   keeps her there.")
print("   ZADD + ZRANGE are both O(log N).")

## 🌍 Filtered Leaderboards: By City and Country

Athletes want to see how they rank *locally*, not just globally.

**Solution**: maintain separate sorted sets per filter:

```
leaderboard:segment:1              ← global
leaderboard:segment:1:country:USA  ← USA only
leaderboard:segment:1:city:SF      ← San Francisco only
```

When an effort is recorded, we update ALL relevant sorted sets.

In [ ]:
def build_filtered_leaderboards(segment_id):
    """
    Build global, country, and city leaderboards for a segment.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT se.user_id, u.city, u.country, MIN(se.elapsed_s) AS best_time
        FROM segment_efforts se
        JOIN users u ON se.user_id = u.id
        WHERE se.segment_id = %s
        GROUP BY se.user_id, u.city, u.country;
    """, (segment_id,))

    # Clear existing keys -- including the global one. `...:{id}:*` does NOT match
    # `...:{id}`, so leaving it out lets a member that no longer exists in Postgres
    # survive a "rebuild" forever.
    r.delete(f"leaderboard:segment:{segment_id}")
    for key in r.keys(f"leaderboard:segment:{segment_id}:*"):
        r.delete(key)

    counts = {'global': 0}
    for row in cur.fetchall():
        uid = str(row['user_id'])
        bt = row['best_time']

        # Global leaderboard (already built, but let's rebuild)
        r.zadd(f"leaderboard:segment:{segment_id}", {uid: bt})
        counts['global'] += 1

        # Country leaderboard
        country_key = f"leaderboard:segment:{segment_id}:country:{row['country']}"
        r.zadd(country_key, {uid: bt})
        counts[row['country']] = counts.get(row['country'], 0) + 1

        # City leaderboard
        city_key = f"leaderboard:segment:{segment_id}:city:{row['city']}"
        r.zadd(city_key, {uid: bt})

    conn.close()
    return counts

# Build filtered leaderboards for Embarcadero Sprint. This rebuilds from Postgres,
# which is exactly the operation that exposes any effort that never got persisted.
counts = build_filtered_leaderboards(segment_id=1)

# Carol's effort from the previous cell has to survive a rebuild-from-Postgres.
assert r.zscore("leaderboard:segment:1", "3") == CAROL_TIME_S, (
    "Carol's effort vanished when the leaderboard was rebuilt from Postgres -- it was "
    "only ever written to Redis"
)
assert r.zscore("leaderboard:segment:1:country:USA", "3") == CAROL_TIME_S, (
    "Carol is on the global leaderboard but missing from her own country's -- the "
    "filtered sets were built from a different source than the global one"
)
print("Built filtered leaderboards for Embarcadero Sprint:")
for k, v in counts.items():
    print(f"  {k}: {v} athletes")
print()

# Show global vs USA vs SF leaderboard
for label, key_suffix in [('🌍 Global', ''), ('🇺🇸 USA', ':country:USA'), ('🏙️ San Francisco', ':city:San Francisco')]:
    key = f"leaderboard:segment:1{key_suffix}"
    top = r.zrange(key, 0, 2, withscores=True)
    print(f"{label} top 3:")
    conn = get_db()
    cur = conn.cursor()
    for i, (uid, score) in enumerate(top, 1):
        cur.execute("SELECT username FROM users WHERE id = %s", (int(uid),))
        username = cur.fetchone()[0]
        t = int(score)
        print(f"  #{i} {username:<10} {t//60}:{t%60:02d}")
    conn.close()
    print()

## 📊 Distance Leaderboards with ZINCRBY

Another type of leaderboard: **total distance** across all activities.

When a user completes an activity, we increment their total distance:

```python
redis.zincrby("leaderboard:distance:run:global", distance_m, user_id)
```

`ZINCRBY` atomically adds to the score — perfect for cumulative stats.

In [ ]:
def build_distance_leaderboard():
    """Build a total-distance leaderboard from all completed activities."""
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""
        SELECT a.user_id, a.type, u.country,
               SUM(a.distance_m) AS total_distance
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
        GROUP BY a.user_id, a.type, u.country;
    """)

    for key in r.keys("leaderboard:distance:*"):
        r.delete(key)

    for user_id, activity_type, country, total_dist in cur.fetchall():
        uid = str(user_id)
        # Global by type
        r.zadd(f"leaderboard:distance:{activity_type.lower()}:global", {uid: float(total_dist)})
        # Country by type
        r.zadd(f"leaderboard:distance:{activity_type.lower()}:{country}", {uid: float(total_dist)})

    conn.close()

build_distance_leaderboard()

# Show top runners by total distance
print("🏃 Top Runners by Total Distance (Global):")
top_runners = r.zrevrange("leaderboard:distance:run:global", 0, 9, withscores=True)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
for i, (uid, dist) in enumerate(top_runners, 1):
    cur.execute("SELECT username, display_name, city FROM users WHERE id = %s", (int(uid),))
    u = cur.fetchone()
    print(f"  #{i:>2} {u['display_name']:<20} {dist/1000:>8.1f} km  ({u['city']})")

print()
print("🚴 Top Cyclists by Total Distance (Global):")
top_riders = r.zrevrange("leaderboard:distance:ride:global", 0, 9, withscores=True)
for i, (uid, dist) in enumerate(top_riders, 1):
    cur.execute("SELECT username, display_name, city FROM users WHERE id = %s", (int(uid),))
    u = cur.fetchone()
    print(f"  #{i:>2} {u['display_name']:<20} {dist/1000:>8.1f} km  ({u['city']})")

conn.close()

In [ ]:
# Demonstrate real-time increment with ZINCRBY.
#
# Same rule as segment efforts: Postgres is the source of truth, the sorted set is a
# derived view. ZINCRBY on its own would make Redis and SQL disagree the moment
# anything rebuilt the leaderboard -- and the SQL-vs-Redis comparison at the end of
# this notebook would quietly be comparing two different datasets.
DEMO_TITLE = 'ZINCRBY demo 10K'
DEMO_DISTANCE_M = 10_000

conn = get_db()
cur = conn.cursor()
# Idempotent: drop the demo activity from any previous run, then resync Redis so
# `before` below is a number Postgres also believes.
cur.execute("DELETE FROM activities WHERE title = %s", (DEMO_TITLE,))
conn.close()
build_distance_leaderboard()

before = r.zscore("leaderboard:distance:run:global", "1")
print("Alice just completed a 10 km run!")
print()
print(f"  Before: {before/1000:.1f} km total")

# 1. Durable write first.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    INSERT INTO activities (user_id, type, state, title, distance_m, duration_s,
                            started_at, completed_at)
    VALUES (1, 'RUN', 'COMPLETE', %s, %s, 3000,
            NOW() - INTERVAL '50 minutes', NOW());
""", (DEMO_TITLE, DEMO_DISTANCE_M))
conn.close()

# 2. Then the atomic increment. No read-modify-write, so concurrent uploads compose.
r.zincrby("leaderboard:distance:run:global", DEMO_DISTANCE_M, "1")
r.zincrby("leaderboard:distance:run:USA", DEMO_DISTANCE_M, "1")

after = r.zscore("leaderboard:distance:run:global", "1")
rank = r.zrevrank("leaderboard:distance:run:global", "1")
print(f"  After:  {after/1000:.1f} km total")
print(f"  Rank:   #{rank + 1}")

# The increment must be exactly the distance we wrote...
assert abs(after - (before + DEMO_DISTANCE_M)) < 1e-6, (
    f"ZINCRBY moved the score from {before} to {after}, expected "
    f"{before + DEMO_DISTANCE_M}"
)
# ...and Redis must still agree with what Postgres now holds.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT SUM(distance_m) FROM activities
    WHERE user_id = 1 AND type = 'RUN' AND state = 'COMPLETE'
""")
sql_total = float(cur.fetchone()[0])
conn.close()
assert abs(after - sql_total) < 1, (
    f"Redis says Alice has run {after:.1f} m, Postgres says {sql_total:.1f} m -- the "
    f"increment was applied to only one of the two stores"
)

print()
print("💡 ZINCRBY is atomic — safe even with millions of concurrent updates, and it")
print("   needs no read-modify-write. But atomic is not the same as durable: the row")
print(f"   in Postgres is what lets you rebuild this set. Both stores agree at "
      f"{sql_total/1000:.1f} km.")

## 📅 Time-Range Leaderboards: Weekly King/Queen of the Mountain

Strava shows leaderboards for **this week**, **this month**, **this year**, and **all time**.

How? A separate sorted set per time bucket:

```
leaderboard:segment:1                       ← all time
leaderboard:segment:1:week:2026-W16         ← ISO week 16 of 2026
leaderboard:segment:1:month:2026-04         ← April 2026
```

When an effort is recorded, we update *every* relevant bucket with a single pipeline.
Old weekly keys expire automatically via Redis TTL — no clean-up job needed.

**Bad**: `SELECT ... FROM segment_efforts WHERE started_at BETWEEN ...`
scans millions of rows on every page view.  
**Best**: pre-aggregate into per-bucket sorted sets; reads are O(log N).

In [ ]:
from datetime import datetime, timedelta

def week_bucket(ts: datetime) -> str:
    """ISO week key like '2026-W16'."""
    y, w, _ = ts.isocalendar()
    return f"{y}-W{w:02d}"

def record_effort_time_bucketed(segment_id, user_id, elapsed_s, when: datetime):
    """Update all-time, weekly, and monthly leaderboards for one segment effort."""
    pipe = r.pipeline()
    # All-time (lowest score = fastest)
    pipe.zadd(f"leaderboard:segment:{segment_id}",
              {str(user_id): elapsed_s}, lt=True)
    # Weekly bucket — expires 90 days after the week ends
    wkey = f"leaderboard:segment:{segment_id}:week:{week_bucket(when)}"
    pipe.zadd(wkey, {str(user_id): elapsed_s}, lt=True)
    pipe.expire(wkey, 90 * 24 * 3600)
    # Monthly bucket — expires 400 days later
    mkey = f"leaderboard:segment:{segment_id}:month:{when:%Y-%m}"
    pipe.zadd(mkey, {str(user_id): elapsed_s}, lt=True)
    pipe.expire(mkey, 400 * 24 * 3600)
    pipe.execute()

# Replay every segment effort from Postgres into time-bucketed sets
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT segment_id, user_id, elapsed_s, started_at
    FROM segment_efforts
    ORDER BY started_at;
""")
n = 0
for segment_id, user_id, elapsed_s, started_at in cur.fetchall():
    record_effort_time_bucketed(segment_id, user_id, elapsed_s, started_at)
    n += 1
conn.close()
print(f"Replayed {n} efforts into time-bucketed leaderboards.")

# Show this week's top 3 on the Embarcadero Sprint.
# Derive "this week" from the newest effort's own timestamp, not from the local
# clock: Postgres is in UTC and your laptop probably isn't, so datetime.now() can
# land in a different ISO week and quietly query an empty bucket.
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT MAX(started_at) FROM segment_efforts WHERE segment_id = 1")
newest_effort_at = cur.fetchone()[0]
conn.close()

this_week = week_bucket(newest_effort_at)
wkey = f"leaderboard:segment:1:week:{this_week}"
top = r.zrange(wkey, 0, 2, withscores=True)

print()
print(f"🏆 Embarcadero Sprint — Week {this_week} top 3:")
if not top:
    print("  (no efforts this week)")
else:
    conn = get_db(); cur = conn.cursor()
    for i, (uid, score) in enumerate(top, 1):
        cur.execute("SELECT username FROM users WHERE id = %s", (int(uid),))
        username = cur.fetchone()[0]
        t = int(score)
        print(f"  #{i} {username:<10} {t//60}:{t%60:02d}")
    conn.close()

# The weekly bucket has to actually contain the effort we just recorded, otherwise
# the bucketing is silently dropping writes.
assert top, f"weekly bucket {wkey} is empty but Carol's effort was recorded in it"
assert r.zscore(wkey, "3") == CAROL_TIME_S, (
    f"Carol's {CAROL_TIME_S}s effort is missing from the {this_week} bucket"
)
# All-time can only ever be at least as fast as any single week.
alltime_best = r.zrange("leaderboard:segment:1", 0, 0, withscores=True)[0][1]
assert alltime_best <= top[0][1], (
    f"all-time best ({alltime_best}) is slower than this week's best ({top[0][1]}) -- "
    f"the buckets have drifted apart"
)

# Key design choice: TTL on weekly keys keeps storage bounded automatically
ttl = r.ttl(wkey)
print()
print(f"💡 TTL on '{wkey}' = {ttl} seconds (~{ttl//86400} days).")
print("   Redis auto-deletes old weekly leaderboards — no cron job needed.")
print("   Caveat: the TTL is refreshed on every write, so a bucket that keeps")
print("   receiving efforts lives longer than 90 days after the week ends.")


## 📈 Scaling Leaderboards: Three Approaches

The Hello Interview breakdown covers three approaches. Let's compare them.

| Approach | Latency | Freshness | Complexity | When to Use |
|----------|---------|-----------|------------|-------------|
| **Naive SQL** | Slow (full table scan) | Real-time | Simple | Prototyping only |
| **Periodic aggregation** | Fast (pre-computed) | Stale (minutes/hours) | Medium | Daily/weekly leaderboards |
| **Redis Sorted Sets** | Fast (O(log N)) | Real-time | Medium | Segment & live leaderboards |

In [ ]:
# Approach 1: Naive SQL (slow at scale)
conn = get_db()
cur = conn.cursor()

start = time.time()
for _ in range(50):
    cur.execute("""
        SELECT u.username, SUM(a.distance_m) AS total_distance
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE' AND a.type = 'RUN'
        GROUP BY u.username
        ORDER BY total_distance DESC
        LIMIT 10;
    """)
    cur.fetchall()
sql_avg = ((time.time() - start) / 50) * 1000
conn.close()

# Approach 3: Redis Sorted Set (fast)
start = time.time()
for _ in range(50):
    r.zrevrange("leaderboard:distance:run:global", 0, 9, withscores=True)
redis_avg = ((time.time() - start) / 50) * 1000

# Latency is only meaningful if both sides are answering the same question.
# Compare the answers before comparing the clocks.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT a.user_id, SUM(a.distance_m) AS total_distance
    FROM activities a
    WHERE a.state = 'COMPLETE' AND a.type = 'RUN'
    GROUP BY a.user_id
    ORDER BY total_distance DESC
    LIMIT 1;
""")
sql_top_user, sql_top_total = cur.fetchone()
conn.close()

redis_top_user, redis_top_total = r.zrevrange(
    "leaderboard:distance:run:global", 0, 0, withscores=True)[0]

print(f"Top runner per Postgres: user {sql_top_user} at {sql_top_total/1000:.1f} km")
print(f"Top runner per Redis:    user {redis_top_user} at {redis_top_total/1000:.1f} km")
assert int(redis_top_user) == sql_top_user and abs(redis_top_total - float(sql_top_total)) < 1, (
    f"Redis and Postgres disagree on the leader: Redis says user {redis_top_user} "
    f"({redis_top_total:.1f} m), Postgres says user {sql_top_user} "
    f"({float(sql_top_total):.1f} m). A cache that answers a different question than "
    f"the database is not faster, it is wrong."
)
print()

print("Leaderboard query latency (50 queries each):")
print(f"  Naive SQL (GROUP BY + ORDER BY): {sql_avg:.2f} ms")
print(f"  Redis Sorted Set (ZREVRANGE):    {redis_avg:.2f} ms")
print(f"  Speedup:                         {sql_avg/redis_avg:.1f}×")
print()
print("💡 Read this honestly: with ~200 activities the SQL side is not slow")
print("   because of data volume — most of that time is planning and round-trip. The")
print("   point is the *shape*: SQL rescans and re-sorts on every read and grows with")
print("   the table, while ZREVRANGE is O(log N + M) on a set kept sorted by writes.")
print("   At 36B+ activities that difference stops being a benchmark and starts being")
print("   the reason the feature exists.")

## 🧹 Cleanup

In [ ]:
# Clean up all Redis keys
r = get_redis()
for pattern in ['leaderboard:*']:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)
        print(f"🧹 Deleted {len(keys)} keys matching '{pattern}'")

# The ZINCRBY section wrote a real activity row -- take it back out so re-running
# the notebook doesn't inflate Alice's totals by 10 km each time.
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM activities WHERE title = 'ZINCRBY demo 10K'")
print(f"🧹 Deleted {cur.rowcount} demo activity row(s)")
conn.close()
print("🧹 Done!")
print()
print("Note: Carol's segment effort (segment 1, activity 3) is left in place — it is")
print("real data, and record_segment_effort() is idempotent, so re-running is safe.")

## 📚 Summary

### Key Takeaways

1. **Segments** are defined by start/end GPS points — athletes race through them
2. **Route matching** checks if a GPS trace passes near segment endpoints — and then
   checks that the distance *travelled* between them looks like the segment, because
   endpoints alone will happily credit a detour you never ran
3. Pick the **nearest** point to each endpoint, not the first one inside the threshold —
   otherwise your Python and your SQL give different answers for the same effort
4. **PostGIS ST_DWithin** makes proximity checks fast with spatial indexes, but
   `start_matches JOIN end_matches` is a cross join: use `DISTINCT ON`
5. **Redis Sorted Sets** are the perfect data structure for leaderboards:
   - `ZADD` — add or update a score; `lt=True` makes "keep the best time" a single
     atomic op instead of a lost-update race
   - `ZRANGE` / `ZREVRANGE` — get top/bottom N
   - `ZRANK` — find a user's position
   - `ZINCRBY` — atomically increment a score
6. **Write through Postgres first.** The sorted set is a derived view; an effort that
   only exists in Redis disappears the next time anything rebuilds from the database
7. **Filtered leaderboards** (by city/country) use separate sorted sets
8. At scale, Redis sorted sets are orders of magnitude faster than SQL GROUP BY

### What This Toy Does *Not* Do

- Segments are two points and a length. Real segments carry a full polyline, and
  matching is a curve-similarity problem, not two proximity tests.
- No direction check beyond "the end came after the start", so an out-and-back road
  can still match the wrong way round.
- No flagging: no speed sanity check, so a car driving the segment gets a KOM.
- Redis holds no durable copy of the ranking. Losing the instance means replaying
  every effort out of Postgres — fine here, a real operation at 36B activities.

### System Design Interview Tips

- Start with the simple SQL approach, then explain why it won't scale
- Propose Redis Sorted Sets as the scalable solution
- Mention filtered leaderboards (separate keys per country/city)
- For time-range filtering, combine sorted sets with hashes
- Always discuss the consistency trade-off between Redis and the primary database